# MM-CAD sketch encoder: reference example + training-image framing

Produces `sketch_reference/` (sketch PNG, model-ready input tensor, 768-d embedding) for `sketch_encoder_v1.pth`, using the exact training validation transform now in `inference.image_transform`. Also reports the size/framing of the training sketches.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!git clone -q https://github.com/exanos/MMCAD.git /content/MMCAD 2>/dev/null || (cd /content/MMCAD && git pull -q)
import sys, os, io, zipfile, random, numpy as np, torch; sys.path.insert(0, '/content/MMCAD')
import inference as I
from PIL import Image
from huggingface_hub import hf_hub_download
DRIVE = '/content/drive/MyDrive/MMCAD'
OUT = '/content/sketch_reference'; os.makedirs(OUT, exist_ok=True)
ck = hf_hub_download('exanos/MMCAD', 'checkpoints/sketch_encoder_v1.pth', repo_type='dataset')
enc = I.ViTSketchEncoder()
sd = torch.load(ck, map_location='cpu', weights_only=False); sd = sd.get('model_state_dict', sd)
print(enc.load_state_dict(sd, strict=True)); enc.eval()

In [ ]:
# Training sketches: size + foreground framing (from sketches.zip on Drive)
z = zipfile.ZipFile(f'{DRIVE}/sketches.zip')
names = [n for n in z.namelist() if n.lower().endswith('.png')]
random.seed(0); sample = random.sample(names, 300)
sizes, fw, fh, margin = {}, [], [], []
for n in sample:
    im = Image.open(io.BytesIO(z.read(n))).convert('L'); w, h = im.size
    sizes[(w, h)] = sizes.get((w, h), 0) + 1
    a = np.asarray(im) < 250
    if a.any():
        ys, xs = np.where(a)
        fw.append((xs.max()-xs.min()+1)/w); fh.append((ys.max()-ys.min()+1)/h)
        margin.append(min(xs.min(), ys.min(), w-1-xs.max(), h-1-ys.max())/max(w, h))
q = lambda v: np.percentile(v, [5, 50, 95]).round(3).tolist()
print('n png', len(names), '| sizes', sizes)
print('fg width frac p5/50/95', q(fw), '| fg height frac', q(fh), '| min margin frac', q(margin))
print('bg pixel value (corner) example', np.asarray(Image.open(io.BytesIO(z.read(sample[0]))).convert('RGB'))[0, 0])

In [ ]:
# Reference sketches: PNG + model-ready tensor + embeddings
tf = I.image_transform(I.CONFIG.VIT_MODEL)
for n in sample[:3]:
    stem = os.path.splitext(os.path.basename(n))[0]
    img = Image.open(io.BytesIO(z.read(n))).convert('RGB'); img.save(f'{OUT}/{stem}.png')
    x = tf(I.pad_to_square(img)).unsqueeze(0)
    with torch.no_grad(): zr = enc(x)
    np.savez(f'{OUT}/{stem}.npz', input=x[0].numpy(), embedding_raw=zr[0].numpy(),
             embedding=torch.nn.functional.normalize(zr, dim=-1)[0].numpy())
    e2 = I.embed_image(enc, f'{OUT}/{stem}.png', I.CONFIG.VIT_MODEL, device='cpu')
    print(stem, img.size, 'embed_image vs manual cos', float((e2[0] * torch.nn.functional.normalize(zr, dim=-1)[0]).sum()))

In [ ]:
# Column 15 in TRAINING data: distinct values after float32 normalization
import h5py
st = np.load(hf_hub_download('exanos/MMCAD', 'checkpoints/brep_norm_stats_v4.npz', repo_type='dataset'))
with h5py.File(f'{DRIVE}/brep_features.h5', 'r') as f:
    ff = f['face_features'][:20000]; m = f['face_masks'][:20000] > 0
v = ff[m]
nz = ((v - st['ff_mean']) / st['ff_std']).astype(np.float32)
print('faces', len(v), '| col15 distinct normalized', np.unique(nz[:, 15]).size,
      '| faces not at the mode', int((nz[:, 15] != np.median(nz[:, 15])).sum()))
for c in range(16): print(c, 'distinct', np.unique(nz[:, c]).size, 'p1/p99', np.percentile(nz[:, c], [1, 99]).round(6))

In [ ]:
from google.colab import userdata; from huggingface_hub import HfApi
api = HfApi(token=userdata.get('HF_TOKEN'))
api.upload_folder(folder_path=OUT, path_in_repo='checkpoints/sketch_reference', repo_id='exanos/MMCAD', repo_type='dataset', commit_message='Add sketch encoder reference inputs and embeddings')
print('UPLOAD_OK', [f for f in api.list_repo_files('exanos/MMCAD', repo_type='dataset') if 'sketch_reference' in f])